# Estrutura de dados

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import random
import time
import csv


class Pod:
    def __init__(self, resource_usage, index, errors):
        self.resource_usage = resource_usage
        self.index = index
        self.errors = errors


class Node:
    def __init__(self, capacity, index, opening_cost, allocation_cost, error_penalization, node_affinity):
        self.capacity = capacity
        self.index = index
        self.opening_cost = opening_cost
        self.allocation_cost = allocation_cost
        self.error_penalization = error_penalization
        self.node_affinity = node_affinity
        self.pods = []

    def can_allocate(self, pod):
        return sum(p.resource_usage for p in self.pods) + pod.resource_usage <= self.capacity

    def allocate(self, pod):
        self.pods.append(pod)

    def clear(self):
        self.pods.clear()


class Instance:
    def __init__(self, num_pods, num_nodes, seed=100):
        self.num_pods = num_pods
        self.num_nodes = num_nodes
        self.nodes = []
        self.pods = []
        self.rng = random.Random(seed)

    def create_nodes(self):
        self.nodes = []
        for i in range(self.num_nodes):
            node = Node(
                capacity=self.rng.randint(self.num_pods // self.num_nodes + 1, self.num_pods * 2),
                index=i,
                opening_cost=self.rng.randint(1, 4 * self.num_nodes),
                allocation_cost=self.rng.randint(1, 4 * self.num_nodes),
                error_penalization=self.rng.randint(1, 10),
                node_affinity=self.rng.choice([True, False])
            )
            self.nodes.append(node)

    def create_pods(self):
        self.pods = []
        for j in range(self.num_pods):
            pod = Pod(
                resource_usage=self.rng.randint(1, 10),
                index=j,
                errors=self.rng.randint(1, 20)
            )
            self.pods.append(pod)

    def modify_number_pods(self):
        min_value = int(self.num_pods * 0.05)
        max_value = int(self.num_pods * 0.30)
        modification = self.rng.randint(min_value, max_value)
        if self.rng.choice([True, False]):
            for i in range(modification):
                self.pods.append(Pod(self.rng.randint(1, 10), self.num_pods + i, self.rng.randint(1, 20)))
            self.num_pods += modification
        else:
            for _ in range(modification):
                if self.pods:
                    idx = self.rng.randint(0, len(self.pods) - 1)
                    self.pods.pop(idx)
                    self.num_pods -= 1


class KubeScheduler:
    def __init__(self, nodes):
        self.nodes = nodes

    def schedule_pod(self, pod):
        for node in self.nodes:
            if node.can_allocate(pod):
                node.allocate(pod)
                return node
        return None


def run_kube_scheduler(instance, writer, number_executions=10, total_time=20):
    for time_slot in range(total_time):
        pods = instance.pods
        nodes = instance.nodes
        for node in nodes:
            node.clear()

        scheduler = KubeScheduler(nodes)
        allocation = {}
        opened_nodes = set()

        start_time = time.time()

        for pod in pods:
            allocated_node = scheduler.schedule_pod(pod)
            if allocated_node:
                allocation[pod] = allocated_node
                opened_nodes.add(allocated_node)

        end_time = time.time()
        elapsed_time = (end_time - start_time) * 1000 / number_executions
        used_nodes = len(opened_nodes)

        total_cost = sum(n.opening_cost for n in opened_nodes)
        for pod in pods:
            node = allocation.get(pod)
            if node:
                total_cost += node.allocation_cost + node.error_penalization * pod.errors

        writer.writerow([time_slot, len(pods), used_nodes, round(total_cost, 2), round(elapsed_time, 2), "N/A"])
        instance.modify_number_pods()


def run_exact_optimization(instance, writer, number_executions=10, total_time=20):
    for time_slot in range(total_time):
        pods = instance.pods
        nodes = instance.nodes
        num_pods = len(pods)
        num_nodes = len(nodes)

        alpha = [n.opening_cost for n in nodes]
        beta = [n.allocation_cost for n in nodes]
        gamma = [n.error_penalization for n in nodes]
        U = [n.capacity for n in nodes]
        u = [p.resource_usage for p in pods]
        e = [p.errors for p in pods]

        try:
            env = gp.Env(empty=True)
            env.setParam("OutputFlag", 0)
            env.start()
            model = gp.Model("nodePodsAllocation", env=env)
            model.setParam("MIPGap", 0.03)

            x = model.addVars(num_nodes, vtype=GRB.BINARY, name="x")
            y = model.addVars(num_nodes, num_pods, vtype=GRB.BINARY, name="y")

            obj = gp.LinExpr()
            for i in range(num_nodes):
                obj += alpha[i] * x[i]
                for j in range(num_pods):
                    obj += beta[i] * y[i, j] + gamma[i] * e[j] * y[i, j]

            model.setObjective(obj, GRB.MINIMIZE)

            model.addConstr(x.sum() >= 1, "MinNodes")
            for i in range(num_nodes):
                for j in range(num_pods):
                    model.addConstr(y[i, j] <= x[i], f"AllocIfOpen_{i}_{j}")
            for j in range(num_pods):
                model.addConstr(gp.quicksum(y[i, j] for i in range(num_nodes)) == 1, f"ServePod_{j}")
            for i in range(num_nodes):
                model.addConstr(gp.quicksum(u[j] * y[i, j] for j in range(num_pods)) <= U[i] * x[i], f"CapNode_{i}")

            start = time.time()
            for _ in range(number_executions):
                model.optimize()
            end = time.time()

            elapsed_time = (end - start) * 1000 / number_executions
            used_nodes = sum(1 for i in range(num_nodes) if x[i].X > 0.5)

            mip_gap = model.MIPGap if model.SolCount > 0 else None

            writer.writerow([time_slot, num_pods, used_nodes, model.ObjVal, round(elapsed_time, 2), round(mip_gap, 6) if mip_gap is not None else "N/A"])

        except gp.GurobiError as e:
            print(f"Gurobi Error {e.errno}: {e.message}")
        finally:
            model.dispose()
            env.dispose()

        instance.modify_number_pods()


def run_exact_penalty_optimization(instance, writer, number_executions=10, total_time=20, penalty_weight=1000):
    previous_allocation = None

    for time_slot in range(total_time):
        pods = instance.pods
        nodes = instance.nodes
        num_pods = len(pods)
        num_nodes = len(nodes)

        alpha = [n.opening_cost for n in nodes]
        beta = [n.allocation_cost for n in nodes]
        U = [n.capacity for n in nodes]
        u = [p.resource_usage for p in pods]

        bar_y = [[0 for _ in range(num_pods)] for _ in range(num_nodes)]
        if previous_allocation is not None:
            old_num_nodes = len(previous_allocation)
            old_num_pods = len(previous_allocation[0]) if old_num_nodes > 0 else 0
            for i in range(min(num_nodes, old_num_nodes)):
                for j in range(min(num_pods, old_num_pods)):
                    bar_y[i][j] = previous_allocation[i][j]

        try:
            env = gp.Env(empty=True)
            env.setParam("OutputFlag", 0)
            env.start()
            model = gp.Model("penalizedAllocation", env=env)
            model.setParam("MIPGap", 0.03)

            x = model.addVars(num_nodes, vtype=GRB.BINARY, name="x")
            y = model.addVars(num_nodes, num_pods, vtype=GRB.BINARY, name="y")

            obj = gp.LinExpr()
            for i in range(num_nodes):
                obj += alpha[i] * x[i]
                for j in range(num_pods):
                    obj += beta[i] * y[i, j]
                    if bar_y[i][j] == 1:
                        obj += (1 - y[i, j]) * penalty_weight

            model.setObjective(obj, GRB.MINIMIZE)

            model.addConstr(x.sum() >= 1, "MinNodes")
            for i in range(num_nodes):
                for j in range(num_pods):
                    model.addConstr(y[i, j] <= x[i], f"AllocIfOpen_{i}_{j}")
            for j in range(num_pods):
                model.addConstr(gp.quicksum(y[i, j] for i in range(num_nodes)) == 1, f"ServePod_{j}")
            for i in range(num_nodes):
                model.addConstr(gp.quicksum(u[j] * y[i, j] for j in range(num_pods)) <= U[i] * x[i], f"CapNode_{i}")

            start = time.time()
            for _ in range(number_executions):
                model.optimize()
            end = time.time()

            elapsed_time = (end - start) * 1000 / number_executions
            used_nodes = sum(1 for i in range(num_nodes) if x[i].X > 0.5)

            mip_gap = model.MIPGap if model.SolCount > 0 else None

            writer.writerow([
                time_slot,
                num_pods,
                used_nodes,
                model.ObjVal,
                round(elapsed_time, 2),
                round(mip_gap, 6) if mip_gap is not None else "N/A"
            ])

            previous_allocation = [[int(y[i, j].X > 0.5) for j in range(num_pods)] for i in range(num_nodes)]

        except gp.GurobiError as e:
            print(f"Gurobi Error {e.errno}: {e.message}")
        finally:
            model.dispose()
            env.dispose()

        instance.modify_number_pods()


Instância 200_100 salva em: instances_data/instance_200_100.csv
Instância 200_200 salva em: instances_data/instance_200_200.csv
Instância 500_100 salva em: instances_data/instance_500_100.csv
Instância 500_200 salva em: instances_data/instance_500_200.csv
Instância 1000_100 salva em: instances_data/instance_1000_100.csv
Instância 1000_200 salva em: instances_data/instance_1000_200.csv
Instância 5000_100 salva em: instances_data/instance_5000_100.csv
Instância 5000_200 salva em: instances_data/instance_5000_200.csv
Instância 10000_100 salva em: instances_data/instance_10000_100.csv
Instância 10000_200 salva em: instances_data/instance_10000_200.csv


## Geração de Instâncias

In [ ]:
def generate_instance_evolution(num_pods, num_nodes, total_time=20, output_dir="instances_data"):
    os.makedirs(output_dir, exist_ok=True)
    instance_id = f"{num_pods}_{num_nodes}"
    file_path = os.path.join(output_dir, f"instance_{instance_id}.csv")

    instance = Instance(num_pods, num_nodes)
    instance.create_pods()

    with open(file_path, "w", newline="") as file:
        writer = csv.writer(file, delimiter=";")
        writer.writerow(["time_slot", "pod_index", "resource_usage"])

        for time_slot in range(total_time):
            for pod in instance.pods:
                writer.writerow([time_slot, pod.index, pod.resource_usage])
            instance.modify_number_pods()

    print(f"Instância {instance_id} salva em: {file_path}")

if __name__ == "__main__":
    # Lista de tamanhos que você deseja gerar
    tamanhosPods = [10, 50, 100, 200, 500, 1000, 5000, 10000]
    tamanhosNodes = [10, 20, 50, 100, 200]
    total_time = 20

    for num_pods in tamanhosPods:
        for num_nodes in tamanhosNodes:
            generate_instance_evolution(num_pods, num_nodes, total_time)

## Testando todos os métodos

In [ ]:
import csv
from your_module import Pod, Instance, Node, KubeScheduler, run_exact_optimization, run_exact_penalty_optimization  # ajuste os imports

def load_pods_from_file(file_path, time_slot):
    pods = []
    with open(file_path, newline="") as file:
        reader = csv.DictReader(file, delimiter=";")
        for row in reader:
            if int(row["time_slot"]) == time_slot:
                pods.append(Pod(
                    resource_usage=int(row["resource_usage"]),
                    index=int(row["pod_index"]),
                    errors=0  # Erros são ignorados
                ))
    return pods

def run_kube_scheduler_on_data(pods, num_nodes, writer):
    nodes = [Node(capacity=1000, index=i, opening_cost=10, allocation_cost=5,
                  error_penalization=0, node_affinity=False) for i in range(num_nodes)]
    for node in nodes:
        node.clear()

    scheduler = KubeScheduler(nodes)
    allocation = {}
    opened_nodes = set()

    for pod in pods:
        node = scheduler.schedule_pod(pod)
        if node:
            allocation[pod] = node
            opened_nodes.add(node)

    used_nodes = len(opened_nodes)
    total_cost = sum(n.opening_cost for n in opened_nodes)
    for pod in pods:
        node = allocation.get(pod)
        if node:
            total_cost += node.allocation_cost

    writer.writerow([len(pods), used_nodes, round(total_cost, 2), "N/A"])

def evaluate_all_methods(instance_file, num_nodes, total_time=20, number_executions=10):
    base_name = instance_file.split("/")[-1].replace(".csv", "")
    
    # MÉTODO 1: KubeScheduler
    with open(f"results/kubescheduler_{base_name}.csv", "w", newline="") as f1:
        writer = csv.writer(f1, delimiter=";")
        writer.writerow(["number of pods", "used nodes", "solution cost", "gap"])
        for t in range(total_time):
            pods = load_pods_from_file(instance_file, t)
            run_kube_scheduler_on_data(pods, num_nodes, writer)

    # MÉTODO 2: Exato clássico
    with open(f"results/exact_{base_name}.csv", "w", newline="") as f2:
        writer = csv.writer(f2, delimiter=";")
        writer.writerow(["time slot", "number of pods", "used nodes", "solution cost", "time (ms)", "gap"])
        instance = Instance(0, num_nodes)  # criamos só com os nodes fixos
        instance.create_nodes()
        for t in range(total_time):
            instance.pods = load_pods_from_file(instance_file, t)
            instance.num_pods = len(instance.pods)
            run_exact_optimization(instance, writer, number_executions=number_executions, total_time=1)

    # MÉTODO 3: Exato com penalidade
    with open(f"results/exact_penalty_{base_name}.csv", "w", newline="") as f3:
        writer = csv.writer(f3, delimiter=";")
        writer.writerow(["time slot", "number of pods", "used nodes", "solution cost", "time (ms)", "gap"])
        instance = Instance(0, num_nodes)
        instance.create_nodes()
        run_exact_penalty_optimization(instance, writer, number_executions=number_executions, total_time=total_time, penalty_weight=1000, input_file=instance_file)



In [7]:
import gurobipy as gp
from gurobipy import GRB
import random
import time
import csv
import os

class Pod:
    def __init__(self, resource_usage, index, errors=0):
        self.resource_usage = resource_usage
        self.index = index
        self.errors = errors


class Node:
    def __init__(self, capacity, index, opening_cost, allocation_cost, error_penalization, node_affinity):
        self.capacity = capacity
        self.index = index
        self.opening_cost = opening_cost
        self.allocation_cost = allocation_cost
        self.error_penalization = error_penalization
        self.node_affinity = node_affinity
        self.pods = []

    def can_allocate(self, pod):
        return sum(p.resource_usage for p in self.pods) + pod.resource_usage <= self.capacity

    def allocate(self, pod):
        self.pods.append(pod)

    def clear(self):
        self.pods.clear()


class Instance:
    def __init__(self, num_nodes, seed=100):
        self.num_nodes = num_nodes
        self.nodes = []
        self.rng = random.Random(seed)

    def create_nodes(self):
        self.nodes = []
        for i in range(self.num_nodes):
            node = Node(
                capacity=self.rng.randint(50, 500),
                index=i,
                opening_cost=self.rng.randint(1, 4 * self.num_nodes),
                allocation_cost=self.rng.randint(1, 4 * self.num_nodes),
                error_penalization=0,
                node_affinity=False
            )
            self.nodes.append(node)


def load_pods_from_file(file_path, time_slot):
    pods = []
    with open(file_path, newline="") as file:
        reader = csv.DictReader(file, delimiter=";")
        for row in reader:
            if int(row["time_slot"]) == time_slot:
                pods.append(Pod(
                    resource_usage=int(row["resource_usage"]),
                    index=int(row["pod_index"])
                ))
    return pods


def run_kube_scheduler_on_data(pods, nodes, writer, number_executions=10):
    for node in nodes:
        node.clear()

    scheduler = KubeScheduler(nodes)
    allocation = {}
    opened_nodes = set()

    start_time = time.time()

    for pod in pods:
        allocated_node = scheduler.schedule_pod(pod)
        if allocated_node:
            allocation[pod] = allocated_node
            opened_nodes.add(allocated_node)

    end_time = time.time()
    elapsed_time = (end_time - start_time) * 1000 / number_executions
    used_nodes = len(opened_nodes)

    total_cost = sum(n.opening_cost for n in opened_nodes)
    for pod in pods:
        node = allocation.get(pod)
        if node:
            total_cost += node.allocation_cost

    writer.writerow([len(pods), used_nodes, round(total_cost, 2), round(elapsed_time, 2), "N/A"])


class KubeScheduler:
    def __init__(self, nodes):
        self.nodes = nodes

    def schedule_pod(self, pod):
        for node in self.nodes:
            if node.can_allocate(pod):
                node.allocate(pod)
                return node
        return None


def main():
    instance_files = os.listdir("instances_data")
    total_time = 20
    number_executions = 10

    for file in instance_files:
        if not file.endswith(".csv"):
            continue

        parts = file.replace("instance_", "").replace(".csv", "").split("_")
        num_pods = int(parts[0])
        num_nodes = int(parts[1])

        input_file = os.path.join("instances_data", file)

        instance = Instance(num_nodes)
        instance.create_nodes()

        with open(f"results/kubescheduler_{num_pods}_{num_nodes}.csv", "w", newline="") as f_kube:
            writer = csv.writer(f_kube, delimiter=";")
            writer.writerow(["number of pods", "used Nodes", "solution cost", "time (ms)", "gap"])
            for t in range(total_time):
                pods = load_pods_from_file(input_file, t)
                run_kube_scheduler_on_data(pods, instance.nodes, writer, number_executions)


if __name__ == "__main__":
    os.makedirs("results", exist_ok=True)
    main()
